# Train a motion diffusion model from scratch

**Track A · Human Modeling** · maps to lesson A7 (motion diffusion).

A real **DDPM**: we corrupt motion trajectories with noise over many steps, train an MLP to predict that noise, then *reverse* the process to generate brand-new motions. Same recipe as the **Human Motion Diffusion Model (MDM)** — here on 2D trajectories so it is self-contained and trains in a minute.

> CPU is fine; a GPU makes it instant.

In [ ]:
import os, math, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 4000)); T_LEN = 32
print("device:", device, "| steps:", STEPS)

## 1 · Data — looping 2D motion trajectories (varied radius, phase, noise)

In [ ]:
def make_motion(n):
    t = torch.linspace(0, 2 * math.pi, T_LEN)
    rx = 0.3 + 0.6 * torch.rand(n, 1); ry = 0.3 + 0.6 * torch.rand(n, 1); ph = 2 * math.pi * torch.rand(n, 1)
    x = rx * torch.cos(t + ph); y = ry * torch.sin(t + ph)
    return torch.stack([x, y], -1).reshape(n, -1)          # (n, T_LEN*2)
data = make_motion(4096).to(device); D = data.shape[1]
fig, ax = plt.subplots(1, 5, figsize=(11, 2.4))
for i in range(5):
    m = data[i].reshape(T_LEN, 2).cpu(); ax[i].plot(m[:, 0], m[:, 1]); ax[i].set_aspect("equal"); ax[i].axis("off")
fig.suptitle("real motions"); plt.show()

## 2 · Diffusion schedule + a time-conditioned denoiser

In [ ]:
Tdiff = 200
betas = torch.linspace(1e-4, 0.02, Tdiff, device=device)
alphas = 1 - betas; acp = torch.cumprod(alphas, 0)

def q_sample(x0, t, noise):
    return acp[t].sqrt()[:, None] * x0 + (1 - acp[t]).sqrt()[:, None] * noise

class Denoiser(nn.Module):
    def __init__(self, D, H=256):
        super().__init__()
        self.tef = nn.Sequential(nn.Linear(1, H), nn.SiLU(), nn.Linear(H, H))
        self.net = nn.Sequential(nn.Linear(D + H, H), nn.SiLU(), nn.Linear(H, H), nn.SiLU(), nn.Linear(H, D))
    def forward(self, x, t):
        te = self.tef(t[:, None].float() / Tdiff)
        return self.net(torch.cat([x, te], -1))

## 3 · Train — predict the added noise

In [ ]:
import math, copy
model = Denoiser(D).to(device); opt = torch.optim.Adam(model.parameters(), 2e-4); hist = []
ema = copy.deepcopy(model)
for p in ema.parameters(): p.requires_grad_(False)
for step in range(STEPS + 1):
    for g in opt.param_groups: g["lr"] = 2e-4 * (0.1 + 0.45 * (1 + math.cos(math.pi * step / max(1, STEPS))))
    x0 = data[torch.randint(0, data.shape[0], (256,), device=device)]
    t = torch.randint(0, Tdiff, (256,), device=device); noise = torch.randn_like(x0)
    loss = ((model(q_sample(x0, t, noise), t) - noise) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():                                            # EMA of weights → smoother samples
        for pe, pm in zip(ema.parameters(), model.parameters()): pe.mul_(0.999).add_(pm, alpha=0.001)
    if step % max(1, STEPS // 10) == 0:
        hist.append((step, round(loss.item(), 4))); print(f"step {step:5d}  loss {loss.item():.4f}")
model.load_state_dict(ema.state_dict())                             # sample/save from the EMA weights

## 4 · Sample — reverse diffusion makes new motions

In [ ]:
@torch.no_grad()
def sample(n):
    x = torch.randn(n, D, device=device)
    for ti in reversed(range(Tdiff)):
        t = torch.full((n,), ti, device=device, dtype=torch.long)
        eps = model(x, t)
        mean = (x - betas[ti] / (1 - acp[ti]).sqrt() * eps) / alphas[ti].sqrt()
        x = mean + (betas[ti].sqrt() * torch.randn_like(x) if ti > 0 else 0)
    return x
gen = sample(5).reshape(5, T_LEN, 2).cpu()

## 5 · Compare — generated motions + the training curve

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(13, 2.4))
for i in range(5):
    ax[i].plot(gen[i, :, 0], gen[i, :, 1], "C1"); ax[i].set_aspect("equal"); ax[i].axis("off"); ax[i].set_title("gen")
ax[5].plot(*zip(*hist), "-o"); ax[5].set_title("loss ↓"); ax[5].set_xlabel("step"); ax[5].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/A_motion_diffusion/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/A_motion_diffusion"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/denoiser.pt")
json.dump({"loss": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Same diffusion idea as **LM** generation; supplies motion priors for **AG** agents.

### Where to go next
- Replace 2D trajectories with **SMPL joint-angle sequences** to generate real human motion.
- Add **text conditioning** (a sentence embedding into the denoiser) → **[MDM](https://github.com/GuyTevet/motion-diffusion-model)** (text-to-motion).
- Swap the MLP for a transformer denoiser for longer, higher-quality sequences.